# 07 — Genie Space (NHL Analyst)

Creates a Genie space scoped to the gold layer with hockey-specific
instructions. The instructions block is what makes the answers correct —
SportLogiq metric short-keys, period structure, strength state vocab, and
the shorthand-vs-team_id mapping all live there.

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

In [ ]:
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
SQL_WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")
if not SQL_WAREHOUSE_ID:
    raise ValueError("SQL_WAREHOUSE_ID not set in .env")
print(G)

In [ ]:
table_names = sorted([
    f"{G}.dim_team", f"{G}.dim_player", f"{G}.dim_venue", f"{G}.dim_game", f"{G}.dim_date",
    f"{G}.fact_game_events", f"{G}.fact_player_shifts",
    f"{G}.fact_player_game_metrics", f"{G}.fact_player_season_metrics",
    f"{G}.v_team_standings", f"{G}.v_player_season_leaders", f"{G}.v_shot_map",
])

genie_config = {
    "version": 2,
    "data_sources": {"tables": [{"identifier": t} for t in table_names]},
    "instructions": {
        "text_instructions": [
            {"content": [
                "This space answers natural-language questions over an NHL hockey analytics warehouse sourced from the SportLogiq feed.",
                "When users say 'team', 'club', or use a 3-letter code (e.g. 'BOS', 'TOR'), match against dim_team.team_shorthand.",
                "A 'shot' = fact_game_events.is_shot = TRUE. A 'goal' = fact_game_events.is_goal = TRUE. Shooting % = goals / shots.",
                "x_coord and y_coord on fact_game_events are rink coordinates from SportLogiq (centre-ice = 0,0). Use these for shot map / heat map questions.",
                "zone = 'O' is offensive, 'D' is defensive, 'N' is neutral.",
                "fact_player_game_metrics and fact_player_season_metrics are LONG (key/value). To get a metric per player, filter metric_key (e.g. 'corsi_for', 'fenwick_for', 'expected_goals') and select metric_value.",
                "topic_id is a SportLogiq metric_topic identifier. Common topics include scoring, defense, goaltending, shooting, faceoffs.",
                "Standings: use v_team_standings. Always include conference and division when ranking teams.",
                "Period structure: NHL games have 3 regulation periods (1, 2, 3) plus optional overtime period 4. Period 5 is shootout.",
                "Strength state lives on silver_player_toi.strength: 5v5 = even strength, 5v4 / 4v5 = power play / penalty kill.",
                "Always prefer the gold layer (dim_*, fact_*, v_*) over silver tables for analytic questions.",
            ]}
        ]
    },
}
serialized = json.dumps(genie_config)
print(f"Tables: {len(table_names)}; instructions: {len(serialized)} chars")

In [ ]:
GENIE_TITLE = "SportLogiq NHL — Hockey Analyst"
current_user = w.current_user.me()
parent_path  = f"/Workspace/Users/{current_user.user_name}"

existing_id = None
try:
    resp = w.api_client.do("GET", "/api/2.0/genie/spaces")
    for space in resp.get("spaces", []):
        if space.get("title", "").startswith(GENIE_TITLE):
            existing_id = space["space_id"]
            break
except Exception:
    pass

if existing_id:
    space_id = existing_id
    print(f"Reusing existing Genie space: {space_id}")
else:
    response = w.api_client.do(
        "POST", "/api/2.0/genie/spaces",
        body={
            "warehouse_id":     SQL_WAREHOUSE_ID,
            "title":            GENIE_TITLE,
            "description":      "Ask natural-language questions over the SportLogiq NHL hockey analytics warehouse — standings, shot maps, season leaders, possession, and more.",
            "serialized_space": serialized,
            "parent_path":      parent_path,
        },
    )
    space_id = response.get("space_id", response.get("id", "unknown"))
    print(f"Genie space created: {space_id}")

host = os.getenv("DATABRICKS_HOST", "").rstrip("/")
print(f"\nTitle: {GENIE_TITLE}")
print(f"Open:  {host}/genie/rooms/{space_id}")

## Sample questions to try

1. *Which teams lead the league in standings points?*
2. *Show me a shot map for the offensive zone, league-wide.*
3. *Who are the top 10 players by corsi_for this season?*
4. *Which goalies have the best save percentage at home?*
5. *Compare power-play time on ice between the conference leaders.*